# Baaki — Smart Collection Planner (Google Colab)

This notebook lets the owner **enter a money target and deadline**, checks every customer in the CSV, and generates collection-plan options to choose from.

**Demo assumptions:** `expected_days_to_pay` and `payment_probability` are fictional example inputs. This is a collection recommendation/optimization algorithm, **not a trained AI/ML model**. It does not guarantee payment, send messages, or confirm money received.

## What it does
1. Upload a customer CSV.
2. Enter the amount you need and how many days you have (use `7` for one week).
3. Check which customers could pay within that time and exclude paid/paused/disputed records.
4. Generate different customer combinations that can cover the target.
5. Choose an option and download the selected plan as JSON.

## Step 1 — Upload the CSV

In [ ]:
from google.colab import files
import io, json, os
from itertools import combinations, product
from datetime import datetime, timezone
import pandas as pd

uploaded = files.upload()  # Upload Baaki_dummy_customers.csv
if not uploaded:
    raise ValueError("Please upload Baaki_dummy_customers.csv")

csv_filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[csv_filename]))
print(f"Loaded {csv_filename} — {len(df)} customer/invoice rows")
display(df)

Saving Baaki_dummy_customers.csv to Baaki_dummy_customers.csv
Loaded Baaki_dummy_customers.csv — 8 customer/invoice rows


,customer_id,customer_name,invoice_id,outstanding_rupees,expected_days_to_pay,payment_probability,disputed,paused
0,C001,Customer A,INV-001,4500,1,0.70,False,False
1,C002,Customer B,INV-002,2000,1,0.90,False,False
2,C003,Customer C,INV-003,1500,2,0.85,False,False
3,C004,Customer D,INV-004,1000,3,0.80,False,False
4,C005,Customer E,INV-005,750,2,0.60,False,False
5,C006,Customer F,INV-006,1250,3,0.75,False,False
6,C007,Customer G,INV-007,3000,1,0.80,False,False
7,C008,Customer H,INV-008,1800,4,0.75,False,False


## Step 2 — Validate customer data

In [ ]:
required_columns = [
    "customer_id", "customer_name", "invoice_id", "outstanding_rupees",
    "expected_days_to_pay", "payment_probability", "disputed", "paused"
]
missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"CSV is missing columns: {missing}")

for col in ["outstanding_rupees", "expected_days_to_pay", "payment_probability"]:
    df[col] = pd.to_numeric(df[col], errors="raise")

for col in ["disputed", "paused"]:
    df[col] = df[col].astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])

if (df["outstanding_rupees"] < 0).any():
    raise ValueError("Outstanding amounts cannot be negative.")
if (df["expected_days_to_pay"] < 0).any():
    raise ValueError("Expected payment days cannot be negative.")
if ((df["payment_probability"] < 0) | (df["payment_probability"] > 1)).any():
    raise ValueError("payment_probability must be from 0 to 1.")
if df["customer_id"].duplicated().any():
    print("Warning: duplicate customer IDs found. If a customer has multiple invoices, give each invoice a unique row/invoice ID and aggregate correctly before using this model.")

print("Data validation passed.")

Data validation passed.


## Step 3 — Enter your money target and deadline
Type the amount you need, for example `4500`. For a one-week deadline, enter `7`.

In [ ]:
def read_positive_int(prompt, default=None, maximum=10**12):
    while True:
        suffix = f" [{default}]" if default is not None else ""
        raw = input(prompt + suffix + ": ").strip()
        if raw == "" and default is not None:
            value = default
        else:
            try:
                value = int(raw.replace(",", ""))
            except ValueError:
                print("Enter a whole number, without ₹. Example: 4500")
                continue
        if value <= 0 or value > maximum:
            print(f"Enter a number between 1 and {maximum}.")
            continue
        return value

TARGET_RUPEES = read_positive_int("How much money do you need (₹)", default=0)
DEADLINE_DAYS = read_positive_int("Within how many days? (enter 7 for one week)", default=7, maximum=365)

print(f"\nTarget: ₹{TARGET_RUPEES:,}")
print(f"Deadline: within {DEADLINE_DAYS} day(s)")

How much money do you need (₹) [4500]: 7000
Within how many days? (enter 7 for one week) [7]: 2

Target: ₹7,000
Deadline: within 2 day(s)


## Step 4 — Check every customer against the deadline
Eligible means the invoice has money outstanding, is not disputed or paused, and has an expected payment day on or before your deadline.

In [ ]:
df["eligibility_reason"] = "Eligible for plan"
df.loc[df["outstanding_rupees"] <= 0, "eligibility_reason"] = "No outstanding amount"
df.loc[df["disputed"], "eligibility_reason"] = "Disputed — review first"
df.loc[df["paused"] & (df["outstanding_rupees"] > 0) & (~df["disputed"]), "eligibility_reason"] = "Paused — do not schedule reminder"
df.loc[(df["expected_days_to_pay"] > DEADLINE_DAYS) & (df["outstanding_rupees"] > 0) & (~df["disputed"]) & (~df["paused"]), "eligibility_reason"] = "Expected after deadline"

eligible_mask = (
    (df["outstanding_rupees"] > 0)
    & (~df["disputed"])
    & (~df["paused"])
    & (df["expected_days_to_pay"] <= DEADLINE_DAYS)
)
eligible_df = df[eligible_mask].copy().reset_index(drop=True)

print(f"\nChecked all {len(df)} customer rows.")
print(f"Eligible within {DEADLINE_DAYS} day(s): {len(eligible_df)}")
display(df[["customer_name", "invoice_id", "outstanding_rupees", "expected_days_to_pay", "disputed", "paused", "eligibility_reason"]])
print(f"Total eligible amount (if everyone in this list pays): ₹{int(eligible_df['outstanding_rupees'].sum()):,}")


Checked all 8 customer rows.
Eligible within 2 day(s): 5


,customer_name,invoice_id,outstanding_rupees,expected_days_to_pay,disputed,paused,eligibility_reason
0,Customer A,INV-001,4500,1,False,False,Eligible for plan
1,Customer B,INV-002,2000,1,False,False,Eligible for plan
2,Customer C,INV-003,1500,2,False,False,Eligible for plan
3,Customer D,INV-004,1000,3,False,False,Expected after deadline
4,Customer E,INV-005,750,2,False,False,Eligible for plan
5,Customer F,INV-006,1250,3,False,False,Expected after deadline
6,Customer G,INV-007,3000,1,False,False,Eligible for plan
7,Customer H,INV-008,1800,4,False,False,Expected after deadline


Total eligible amount (if everyone in this list pays): ₹11,750


## Step 5 — Generate possible collection plans
The planner tests every combination of eligible customers. Its payment chances are only as good as the probabilities in your CSV; current demo percentages are fictional.

In [ ]:
def probability_target_met(group, target_rupees):
    # Exact chance of collecting at least target if each selected customer
    # either pays their full outstanding balance by deadline or does not pay.
    # Assumes probabilities are correct and outcomes are independent.
    chance = 0.0
    for paid_flags in product([False, True], repeat=len(group)):
        amount = 0
        scenario_p = 1.0
        for customer, pays in zip(group, paid_flags):
            p = float(customer["payment_probability"])
            if pays:
                amount += int(customer["outstanding_rupees"])
                scenario_p *= p
            else:
                scenario_p *= (1 - p)
        if amount >= target_rupees:
            chance += scenario_p
    return chance

customers = eligible_df.to_dict(orient="records")
all_plans = []
for size in range(1, len(customers) + 1):
    for group_tuple in combinations(customers, size):
        group = list(group_tuple)
        total = sum(int(c["outstanding_rupees"]) for c in group)
        if total < TARGET_RUPEES:
            continue
        chance = probability_target_met(group, TARGET_RUPEES)
        all_plans.append({
            "customers": group,
            "customer_count": len(group),
            "customer_names": [c["customer_name"] for c in group],
            "invoice_ids": [c["invoice_id"] for c in group],
            "amount_if_all_pay": total,
            "over_target_by": total - TARGET_RUPEES,
            "probability_target_met": chance,
        })

# Prefer options no more than 25% above the target if any such plans exist.
reasonable = [p for p in all_plans if p["amount_if_all_pay"] <= TARGET_RUPEES * 1.25]
plans_for_options = reasonable if reasonable else all_plans

print(f"Feasible combinations able to cover the target: {len(all_plans)}")
if not all_plans:
    eligible_total = int(eligible_df["outstanding_rupees"].sum())
    print("No combination in the eligible list reaches your target.")
    print(f"Eligible outstanding total: ₹{eligible_total:,}")
    print(f"Amount short if all eligible invoices are collected: ₹{max(0, TARGET_RUPEES - eligible_total):,}")

Feasible combinations able to cover the target: 12


## Step 6 — Display three different recommendations

In [ ]:
def key(plan):
    return tuple(sorted(c["customer_id"] for c in plan["customers"]))

options = []
used = set()
if plans_for_options:
    strategies = [
        ("Fewest customers to contact", sorted(
            plans_for_options,
            key=lambda p: (p["customer_count"], -p["probability_target_met"], p["over_target_by"])
        )),
        ("Highest estimated chance of reaching target", sorted(
            plans_for_options,
            key=lambda p: (-p["probability_target_met"], p["customer_count"], p["over_target_by"])
        )),
        ("Multi-customer plan closest to target", sorted(
            [p for p in plans_for_options if p["customer_count"] >= 2] or plans_for_options,
            key=lambda p: (p["over_target_by"], -p["probability_target_met"], p["customer_count"])
        )),
    ]
    for label, ranked in strategies:
        pick = next((p for p in ranked if key(p) not in used), None)
        if pick is not None:
            options.append({"label": label, "plan": pick})
            used.add(key(pick))
    fallback = sorted(plans_for_options, key=lambda p: (-p["probability_target_met"], p["customer_count"], p["over_target_by"]))
    for p in fallback:
        if len(options) >= min(3, len(plans_for_options)):
            break
        if key(p) not in used:
            options.append({"label": "Alternative feasible plan", "plan": p})
            used.add(key(p))

if options:
    rows = []
    for idx, opt in enumerate(options, start=1):
        p = opt["plan"]
        rows.append({
            "option": idx,
            "strategy": opt["label"],
            "customers": ", ".join(p["customer_names"]),
            "invoices": ", ".join(p["invoice_ids"]),
            "number_of_customers": p["customer_count"],
            "amount_if_all_pay_rupees": p["amount_if_all_pay"],
            "extra_over_target_rupees": p["over_target_by"],
            "demo_chance_target_met_percent": round(p["probability_target_met"] * 100, 1),
        })
    options_df = pd.DataFrame(rows)
    display(options_df)
    print("These probabilities are fictional demo estimates—not validated forecasts or guarantees.")
else:
    print("No plan options were generated. Increase the deadline or review the outstanding data.")

,option,strategy,customers,invoices,number_of_customers,amount_if_all_pay_rupees,extra_over_target_rupees,demo_chance_target_met_percent
0,1,Fewest customers to contact,"Customer A, Customer G","INV-001, INV-007",2,7500,500,56.0
1,2,Highest estimated chance of reaching target,"Customer A, Customer B, Customer C, Customer E","INV-001, INV-002, INV-003, INV-005",4,8750,1750,59.2
2,3,Multi-customer plan closest to target,"Customer A, Customer B, Customer E","INV-001, INV-002, INV-005",3,7250,250,37.8


These probabilities are fictional demo estimates—not validated forecasts or guarantees.


## Step 7 — Owner chooses one plan
This saves a selected plan. It does not send reminders or mark invoices paid.

In [ ]:
if options:
    choice_text = input(f"Choose your plan number (1-{len(options)}): ").strip()
    try:
        choice = int(choice_text)
        if not 1 <= choice <= len(options):
            raise ValueError
    except ValueError:
        print("Invalid choice. No plan saved. Re-run this cell and choose a displayed number.")
        selected_plan = None
    else:
        selected_option = options[choice - 1]
        p = selected_option["plan"]
        saved = {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "demo_only": True,
            "target_rupees": TARGET_RUPEES,
            "deadline_days": DEADLINE_DAYS,
            "selected_option": choice,
            "strategy": selected_option["label"],
            "customers": [
                {
                    "customer_id": c["customer_id"],
                    "customer_name": c["customer_name"],
                    "invoice_id": c["invoice_id"],
                    "outstanding_rupees": int(c["outstanding_rupees"]),
                    "expected_days_to_pay": int(c["expected_days_to_pay"]),
                    "payment_probability_demo": float(c["payment_probability"]),
                }
                for c in p["customers"]
            ],
            "total_if_all_pay_rupees": int(p["amount_if_all_pay"]),
            "extra_over_target_rupees": int(p["over_target_by"]),
            "probability_target_met_demo": round(float(p["probability_target_met"]), 6),
            "status": "selected_not_sent",
            "note": "Dummy data and probabilities only. The owner must verify collections; no reminders sent and no payments verified."
        }
        with open("selected_plan.json", "w", encoding="utf-8") as f:
            json.dump(saved, f, indent=2, ensure_ascii=False)
        print("\nPlan saved to selected_plan.json")
        print("Selected strategy:", selected_option["label"])
        print("Customers:", ", ".join(p["customer_names"]))
        print(f"Total if all selected customers pay: ₹{p['amount_if_all_pay']:,}")
        print("Status: selected_not_sent")
else:
    print("There is no plan to select. Change the target/deadline or inspect the customer data.")

Choose your plan number (1-3): 2

Plan saved to selected_plan.json
Selected strategy: Highest estimated chance of reaching target
Customers: Customer A, Customer B, Customer C, Customer E
Total if all selected customers pay: ₹8,750
Status: selected_not_sent


## Step 8 — Download the selected plan

In [ ]:
if os.path.exists("selected_plan.json"):
    files.download("selected_plan.json")
else:
    print("No saved plan yet. First complete Step 7.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Important limits and next steps
- The current notebook is a **rule-based collection optimizer**, not a trained AI model.
- The payment probabilities in the dummy CSV are made up. Do not use them as real-world estimates.
- It considers expected pay day and payment probability, and assumes each selected customer either pays their full outstanding balance by the deadline or does not pay; it assumes independent outcomes.
- It does not message customers, connect to banks, verify payments, or mark invoices paid.
- To make this a genuine ML system, replace the dummy rows with permissioned historical invoice/payment data, create labels such as `paid_by_deadline`, train and evaluate on later unseen data, and calibrate probabilities before using them for planning.